In [ ]:
"""
Generate one YAML per run.
Each YAML corresponds to exactly one Slurm job.
"""

from pathlib import Path
import yaml
import itertools

def base_config(n_samples=100, n_seqs=-1):
    """Return base config shared by all runs."""
    return {
        "results_root": "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory",
        "tag": "slurm",
        "experiment": {
            "is_multi": True,
            "n_seqs": n_seqs,
            "n_samples": n_samples,
        },
    }



In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
resnet_layers = [
        'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
    ]

time_avg = True

for layer in resnet_layers:

    cur_path = f"../model_yamls/three-regime/{layer}/time_avg"
    OUT_DIR = Path(cur_path)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    tasks = {
        0: "ind-nature",
        1: "global-music",
        2: "atexts",
    }
    
    metrics = ["cosine"]#, "euclidean", "loglikelihood"]
    
    
    epsilon = 0.0005
    
    # model_cfg['noise_model']['sigma0_max'] = 0.4
    # model_cfg['noise_model']['sigma1_max'] = 0.12 
    
    # model_cfg['noise_model']['sigma0_min'] = 0.2
    # model_cfg['noise_model']['sigma1_min'] = 0.0005
    
    noise_models = [
        {
            "name": "three-regime",
            "sigma0_min": 0.7,
            "sigma0_max": 0.75,
            "sigma1_min": 0.1,
            "sigma1_max": 0.6,
            "sigma2_min": 0.0005,
            "sigma2_max": 0.1,
            "t_step": 5,
        },
    ]
    
    
    representations = [
        ("resnet50", {"layer": [layer]}),
       # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
    ]

    # ---------------------------
    # GENERATION
    # ---------------------------
    run_idx = 0
    
    for task_id, task_name in tasks.items():
        for metric in metrics:
            for noise in noise_models:
                for repr_type, repr_params in representations:
    
                    if repr_type == "texture_pca" and task_id in (0, 1):
                        continue
    
                    cfg_base = base_config(n_samples=50, n_seqs=36)
                    cfg_base["experiment"]["which_task"] = task_id
                    cfg_base["metric"] = metric
                    cfg_base["noise_model"] = noise
    
                    # -------- Layer-based representations --------
                    if "layer" in repr_params:
                        for layer in repr_params["layer"]:
                            cfg = cfg_base.copy()
                            cfg["run_id"] = f"run_{run_idx:06d}"
                            cfg["representation"] = {
                                "type": repr_type,
                                "layer": layer,
                                "time_avg": time_avg
                            }
    
                            out = OUT_DIR / f"{cfg['run_id']}.yaml"
                            with open(out, "w") as f:
                                yaml.safe_dump(cfg, f, sort_keys=False)
    
                            run_idx += 1
    
                    # -------- PC-based representations --------
                    if "pc_dims" in repr_params:
                        for pc in repr_params["pc_dims"]:
                            cfg = cfg_base.copy()
                            cfg["run_id"] = f"run_{run_idx:06d}"
                            cfg["representation"] = {
                                "type": repr_type,
                                "pc_dims": pc,
                            }
                            cfg["pc_dims"] = pc
    
                            out = OUT_DIR / f"{cfg['run_id']}.yaml"
                            with open(out, "w") as f:
                                yaml.safe_dump(cfg, f, sort_keys=False)
    
                            run_idx += 1

    print(cur_path)
    print(f"Generated {run_idx} YAML files")
    
    #("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
resnet_layers = [
        'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
    ]

time_avg = True
cur_path = f"../model_yamls/three-regime/resnet50/time_avg"
OUT_DIR = Path(cur_path)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]


epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [
    {
        "name": "three-regime",
        "sigma0_min": 3.0,
        "sigma0_max": 0.5,
        "sigma1_min": 0.1,
        "sigma1_max": 0.6,
        "sigma2_min": 0.0005,
        "sigma2_max": 0.1,
        "t_step": 5,
    },
]


representations = [
    ("resnet50", {"layer": resnet_layers}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                            "time_avg": time_avg
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

print(cur_path)
print(f"Generated {run_idx} YAML files")
    
    #("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
resnet_layers = [
        'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
    ]

time_avg = False
cur_path = f"../model_yamls/three-regime/resnet50/nontime_avg"
OUT_DIR = Path(cur_path)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]


epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [
    {
        "name": "three-regime",
        "sigma0_min": 3.0,
        "sigma0_max": 0.5,
        "sigma1_min": 0.1,
        "sigma1_max": 0.6,
        "sigma2_min": 0.0005,
        "sigma2_max": 0.1,
        "t_step": 5,
    },
]


representations = [
    ("resnet50", {"layer": resnet_layers}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                            "time_avg": time_avg
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

print(cur_path)
print(f"Generated {run_idx} YAML files")
    
    #("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------

kell_layers = [
        'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
    ]

time_avg = True
cur_path = f"../model_yamls/three-regime/kell2018/time_avg"
OUT_DIR = Path(cur_path)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]


epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [

    {
        "name": "three-regime",
        "sigma0_min": 3.0,
        "sigma0_max": 0.5,
        "sigma1_min": 0.1,
        "sigma1_max": 0.6,
        "sigma2_min": 0.0005,
        "sigma2_max": 0.1,
        "t_step": 5,
    },
]


representations = [
    ("kell2018", {"layer": kell_layers}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                            "time_avg": time_avg
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

print(cur_path)
print(f"Generated {run_idx} YAML files")

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------

kell_layers = [
        'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
    ]

time_avg = False
cur_path = f"../model_yamls/three-regime/kell2018/nontime_avg"
OUT_DIR = Path(cur_path)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]


epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [

    {
        "name": "three-regime",
        "sigma0_min": 3.0,
        "sigma0_max": 0.5,
        "sigma1_min": 0.1,
        "sigma1_max": 0.6,
        "sigma2_min": 0.0005,
        "sigma2_max": 0.1,
        "t_step": 5,
    },
]


representations = [
    ("kell2018", {"layer": kell_layers}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                            "time_avg": time_avg
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

print(cur_path)
print(f"Generated {run_idx} YAML files")

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------

pc_dims = [
        64, 128, 256, 512, 1028
    ]

cur_path = f"../model_yamls/three-regime/texture_model"
OUT_DIR = Path(cur_path)
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]


epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [

    {
        "name": "three-regime",
        "sigma0_min": 3.0,
        "sigma0_max": 0.5,
        "sigma1_min": 0.1,
        "sigma1_max": 0.6,
        "sigma2_min": 0.0005,
        "sigma2_max": 0.1,
        "t_step": 5,
    },
]


representations = [
   ("texture_pca", {"pc_dims": pc_dims}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

print(cur_path)
print(f"Generated {run_idx} YAML files")

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------


OUT_DIR = Path("../model_yamls/relu4")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]

#model_yamls
# noise_models = [
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 3,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 4,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 5,
#     },
#     {
#         "name": "constant",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#     },
# ]

epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.02 

# model_cfg['noise_model']['sigma0_min'] = 0.05
# model_cfg['noise_model']['sigma1_min'] = 0.001

noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.05,
        "sigma0_max": 0.4,
        "sigma1_min": 0.001,
        "sigma1_max": 0.02,
        "t_step": 4,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.05,
        "sigma0_max": 0.4,
        "sigma1_min": 0.001,
        "sigma1_max": 0.02,
        "t_step": 5,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.05,
        "sigma0_max": 0.4,
        "sigma1_min": 0.001,
        "sigma1_max": 0.02,
        "t_step": 6,
    },
]


representations = [
    ("kell2018", {"layer": ["relu4"]}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")

#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------


OUT_DIR = Path("../model_yamls/relufc")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]

#model_yamls
# noise_models = [
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 3,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 4,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 5,
#     },
#     {
#         "name": "constant",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#     },
# ]

epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 4,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 5,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 6,
    },
]


representations = [
    ("kell2018", {"layer": ["relufc"]}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")

#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------

OUT_DIR = Path("../model_yamls/input_after_preproc")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]

#model_yamls
# noise_models = [
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 3,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 4,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 5,
#     },
#     {
#         "name": "constant",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#     },
# ]

######"../model_yamls/two-regime-kell2018"
noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 3,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 4,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 5,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 6,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 7,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 8,
    },
    {
        "name": "constant",
        "sigma0_min": 0.0,
        "sigma0_max": 4.0,
    },
]


representations = [
    ("kell2018", {"layer": ["input_after_preproc"]}),
    #("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------


run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=100, n_seqs=24)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:05d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:05d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")

#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------


OUT_DIR = Path("../model_yamls/layer4")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

metrics = ["cosine"]#, "euclidean", "loglikelihood"]

#model_yamls
# noise_models = [
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 3,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 4,
#     },
#     {
#         "name": "two-regime",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#         "sigma1_min": 0.0,
#         "sigma1_max": 1.0,
#         "t_step": 5,
#     },
#     {
#         "name": "constant",
#         "sigma0_min": 0.0,
#         "sigma0_max": 2.0,
#     },
# ]

epsilon = 0.0005

# model_cfg['noise_model']['sigma0_max'] = 0.4
# model_cfg['noise_model']['sigma1_max'] = 0.12 

# model_cfg['noise_model']['sigma0_min'] = 0.2
# model_cfg['noise_model']['sigma1_min'] = 0.0005

noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 4,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 5,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.2,
        "sigma0_max": 0.4,
        "sigma1_min": 0.0005,
        "sigma1_max": 0.12,
        "t_step": 6,
    },
]


representations = [
    ("resnet50", {"layer": ["layer4"]}),
   # ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------
run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=50, n_seqs=36)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")

#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
OUT_DIR = Path("../model_yamls/all-texture-pca")
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics = ["cosine", "euclidean", "loglikelihood"]

######"../model_yamls/two-regime-kell2018"
noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 2,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 3,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 4,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 5,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 6,
    },
    {
        "name": "constant",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
    },
]


representations = [
    #("kell2018", {"layer": ["input_after_preproc", "relu1", "relu2", "relu3", "relu4", "relufc"]}),
    ("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------

run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config()
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")
#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
OUT_DIR = Path("../model_yamls/two-regime-kell2018-t6_t2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics = ["cosine", "euclidean", "loglikelihood"]

######"../model_yamls/two-regime-kell2018"
noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 2,
    },
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 2.0,
        "sigma1_min": 0.0,
        "sigma1_max": 1.0,
        "t_step": 6,
    },
]


representations = [
    ("kell2018", {"layer": ["input_after_preproc", "relu1", "relu2", "relu3", "relu4", "relufc"]}),
    #("texture_pca", {"pc_dims": [256, 512, 1028]}),
]

# ---------------------------
# GENERATION
# ---------------------------

run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config()
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }
                        cfg["pc_dims"] = pc

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")
#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),

In [ ]:
# ---------------------------
# GRIDS (explicit & safe)
# ---------------------------
OUT_DIR = Path("../model_yamls/test")
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics = ["cosine", "euclidean", "loglikelihood"]

######"../model_yamls/two-regime-kell2018"
noise_models = [
    {
        "name": "two-regime",
        "sigma0_min": 0.0,
        "sigma0_max": 1.0,
        "sigma1_min": 0.0,
        "sigma1_max": 0.5,
        "t_step": 3,
    },
    {
        "name": "constant",
        "sigma0_min": 0.0,
        "sigma0_max": 1.0,
    },
]


representations = [
    ("kell2018", {"layer": ["relufc"]}),
    ("texture_pca", {"pc_dims": [512]}),
]

# ---------------------------
# GENERATION
# ---------------------------

run_idx = 0

for task_id, task_name in tasks.items():
    for metric in metrics:
        for noise in noise_models:
            for repr_type, repr_params in representations:

                if repr_type == "texture_pca" and task_id in (0, 1):
                    continue

                cfg_base = base_config(n_samples=10, n_seqs=10)
                cfg_base["experiment"]["which_task"] = task_id
                cfg_base["metric"] = metric
                cfg_base["noise_model"] = noise

                # -------- Layer-based representations --------
                if "layer" in repr_params:
                    for layer in repr_params["layer"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "layer": layer,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1

                # -------- PC-based representations --------
                if "pc_dims" in repr_params:
                    for pc in repr_params["pc_dims"]:
                        cfg = cfg_base.copy()
                        cfg["run_id"] = f"run_{run_idx:06d}"
                        cfg["representation"] = {
                            "type": repr_type,
                            "pc_dims": pc,
                        }

                        out = OUT_DIR / f"{cfg['run_id']}.yaml"
                        with open(out, "w") as f:
                            yaml.safe_dump(cfg, f, sort_keys=False)

                        run_idx += 1
                        
print(f"Generated {run_idx} YAML files")
#("resnet50", {"layer": ["layer1", "layer2", "layer3", "layer4"]}),